In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from brain_image.utils import setup_logging


setup_logging()

In [ ]:
import logging
import pandas as pd
import json
import yaml
from pathlib import Path


def get_single_file(dir: Path, pattern: str) -> Path | None:
    paths = list(dir.rglob(pattern))

    num_results = len(paths)
    if num_results == 0:
        return None

    if num_results > 1:
        raise ValueError(f"Expected to find one results matching pattern {pattern} in dir {dir} - Found {num_results}: {tuple(paths)}")

    path = paths[0]
    return path
        

def gather_metrics(experiment_dir: Path, selected_hparams: list[str] = []) -> pd.DataFrame:
    all_metrics = []

    for exp_dir in experiment_dir.iterdir():
        metrics_path = get_single_file(exp_dir, "*test_metrics.json")
        if metrics_path is None:
            logging.warning(f"Could not find any paths in dir {exp_dir} matching pattern {'*test/test_metrics.json'}")
            continue

        logging.info(f"Loading metrics from {metrics_path}")

        with open(metrics_path, "r") as f:
            metrics = json.load(f)

        if len(selected_hparams) > 0:
            hparams_path = get_single_file(exp_dir, "*hparams.yaml")
            if hparams_path is None:
                logging.warning(f"Could not find hparam file")
                continue
            
            with open(hparams_path) as f:
                hparams = yaml.safe_load(f)

            for hparam_key in selected_hparams:
                hparam_parts = hparam_key.split(".")
                curr_hparam = hparams
                for part in hparam_parts:
                    curr_hparam = curr_hparam[part]

                metrics[hparam_key] = curr_hparam

        all_metrics.append(metrics)

    metrics = pd.DataFrame.from_records(all_metrics)
    return metrics


ex_path = Path("experiments/encoders-full")
metrics = gather_metrics(ex_path, ["config.align_img_encoder", "config.eeg_encoder"])
metrics

12:53:39 | INFO     | Loading metrics from experiments/encoders-full/251031151853lybfym-slurmarr14533684_61/version_0/test_metrics.json


KeyError: 'dataset'